In [3]:
%pip install implicit

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import implicit as imp
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix

In [2]:
df_movie = pd.read_csv(r"C:\Users\Tobis\Desktop\Implicit Recommendation System\movie\movie.csv")
df_rating = pd.read_csv(r"C:\Users\Tobis\Desktop\Implicit Recommendation System\rating\rating.csv")

In [3]:
print(df_rating.head())

   userId  movieId  rating            timestamp
0       1        2     3.5  2005-04-02 23:53:47
1       1       29     3.5  2005-04-02 23:31:16
2       1       32     3.5  2005-04-02 23:33:39
3       1       47     3.5  2005-04-02 23:32:07
4       1       50     3.5  2005-04-02 23:29:40


In [4]:
print(df_movie.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [5]:
# Neue Spalte "implicit" erstellen
df_rating["implicit"] = (df_rating["rating"] >= 4).astype(int)
print(df_rating.head())

   userId  movieId  rating            timestamp  implicit
0       1        2     3.5  2005-04-02 23:53:47         0
1       1       29     3.5  2005-04-02 23:31:16         0
2       1       32     3.5  2005-04-02 23:33:39         0
3       1       47     3.5  2005-04-02 23:32:07         0
4       1       50     3.5  2005-04-02 23:29:40         0


In [6]:
user_ids = df_rating["userId"].unique()
movie_ids = df_rating["movieId"].unique()

user_map = {u: i for i, u in enumerate(user_ids)}
movie_map = {m: i for i, m in enumerate(movie_ids)}

reverse_movie_map = {v: k for k, v in movie_map.items()}

In [7]:
df_positive = df_rating[df_rating["implicit"] == 1].copy()

df_positive["user_idx"] = df_positive["userId"].map(user_map)
df_positive["movie_idx"] = df_positive["movieId"].map(movie_map)

In [8]:
print(df_positive)

          userId  movieId  rating            timestamp  implicit  user_idx  \
6              1      151     4.0  2004-09-10 03:08:54         1         0   
7              1      223     4.0  2005-04-02 23:46:13         1         0   
8              1      253     4.0  2005-04-02 23:35:40         1         0   
9              1      260     4.0  2005-04-02 23:33:46         1         0   
10             1      293     4.0  2005-04-02 23:31:43         1         0   
...          ...      ...     ...                  ...       ...       ...   
20000256  138493    66762     4.5  2009-10-17 18:50:08         1    138492   
20000257  138493    68319     4.5  2009-12-07 18:15:20         1    138492   
20000258  138493    68954     4.5  2009-11-13 15:42:00         1    138492   
20000259  138493    69526     4.5  2009-12-03 18:31:48         1    138492   
20000261  138493    70286     5.0  2009-11-13 15:42:24         1    138492   

          movie_idx  
6                 6  
7                 7

In [34]:
user_item = csr_matrix(
    (
        df_positive["implicit"],
        (
            df_positive["user_idx"],
            df_positive["movie_idx"]
        )
    )
)
print(user_item)

  (0, 6)	1
  (0, 7)	1
  (0, 8)	1
  (0, 9)	1
  (0, 10)	1
  (0, 11)	1
  (0, 12)	1
  (0, 15)	1
  (0, 22)	1
  (0, 23)	1
  (0, 26)	1
  (0, 27)	1
  (0, 30)	1
  (0, 31)	1
  (0, 32)	1
  (0, 35)	1
  (0, 36)	1
  (0, 38)	1
  (0, 40)	1
  (0, 43)	1
  (0, 44)	1
  (0, 45)	1
  (0, 48)	1
  (0, 49)	1
  (0, 52)	1
  :	:
  (138492, 5588)	1
  (138492, 5829)	1
  (138492, 5836)	1
  (138492, 5970)	1
  (138492, 6121)	1
  (138492, 6156)	1
  (138492, 6591)	1
  (138492, 7124)	1
  (138492, 7421)	1
  (138492, 7557)	1
  (138492, 7717)	1
  (138492, 7722)	1
  (138492, 7733)	1
  (138492, 7953)	1
  (138492, 8056)	1
  (138492, 8233)	1
  (138492, 8890)	1
  (138492, 9527)	1
  (138492, 9547)	1
  (138492, 9648)	1
  (138492, 9787)	1
  (138492, 10743)	1
  (138492, 11011)	1
  (138492, 11504)	1
  (138492, 11614)	1


In [10]:
model = AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=20
)

# implicit erwartet Item-User-Matrix
model.fit(user_item.T)

C:\Users\Tobis\Python\anaconda3\lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: Intel MKL BLAS is configured to use 6 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()
C:\Users\Tobis\Python\anaconda3\lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.09529280662536621 seconds
  warnings.warn(


  0%|          | 0/20 [00:00<?, ?it/s]

In [35]:
user_idx = user_map[4]

ids, scores = model.recommend(
    userid=user_idx,
    user_items=user_item[user_idx],
    N=10
)
print(ids)
print(scores)

[ 20131  80500  34418  29692  12728  23665  66998  41908  91866 107755]
[1.8166502 1.7946864 1.758287  1.7543732 1.7490752 1.7304602 1.7190969
 1.7132951 1.6791285 1.6738552]


In [36]:
movie_ids = [reverse_movie_map.get(i, None) for i in ids]
print(movie_ids)

[65702, None, None, None, 26996, 118902, None, None, None, None]


In [37]:
recommendations = pd.DataFrame({
    "movieId": movie_ids,
    "score": scores
})

recommendations = recommendations.merge(
    df_movie[["movieId", "title"]],
    on="movieId",
    how="left"
)

print(recommendations)

    movieId     score                                          title
0   65702.0  1.816650  War Comes to America (Why We Fight, 7) (1945)
1       NaN  1.794686                                            NaN
2       NaN  1.758287                                            NaN
3       NaN  1.754373                                            NaN
4   26996.0  1.749075                              Safe House (1998)
5  118902.0  1.730460                            Zebra Lounge (2001)
6       NaN  1.719097                                            NaN
7       NaN  1.713295                                            NaN
8       NaN  1.679129                                            NaN
9       NaN  1.673855                                            NaN
